In [1]:
import os
import shutil

# Let's inspect the exact structure inside /kaggle/input/datasets/
print("Contents of datasets folder:", os.listdir('/kaggle/input/datasets/'))

Contents of datasets folder: ['emnabouhajeb', 'vaatsalyasrivastava']


In [2]:
import os
import shutil

def fast_copy():
    # Hardcoded exact paths discovered in our previous debugging steps
    src = '/kaggle/input/datasets/vaatsalyasrivastava/raf-ravss/raf_ravss'
    dst = '/kaggle/working/code'
    
    if not os.path.exists(src):
        print(f"Error: Source path not found at {src}. Did the dataset change?")
        return

    # Clean up destination if it exists
    if os.path.exists(dst):
        print("Clearing old working directory...")
        shutil.rmtree(dst)
        
    # Copy files directly without searching
    print("Copying files...")
    shutil.copytree(src, dst)
    print("Files copied successfully to /kaggle/working/code!")

fast_copy()

Copying files...
Files copied successfully to /kaggle/working/code!


In [3]:
%cd /kaggle/working/code

/kaggle/working/code


In [4]:
!pip install -q pesq pystoi opencv-python-headless

  Preparing metadata (setup.py) ... done


In [5]:
import os
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'mvlrs_v1' in dirs:
        print("Your true DATA_ROOT is:", os.path.join(root, 'mvlrs_v1'))
        break

Your true DATA_ROOT is: /kaggle/input/datasets/emnabouhajeb/lrs2-v1/mvlrs_v1


In [6]:
%%writefile /kaggle/working/code/config.py
"""
config.py
Central configuration for RAF-RAVSS (audio-visual target-speaker extraction
on LRS2). Every number here was picked with a 16 GB Colab/Kaggle T4/P100 GPU
and a ~9 hour session limit in mind. Shrink BATCH_SIZE / CHUNK_SECONDS first
if you hit OOM; everything else scales off those two.
"""

import os
import torch

class Config:
    # ---------------- paths ----------------
    # Added /mvlrs_v1 to the end so it correctly locates the main/ split folder
    DATA_ROOT   = "/kaggle/input/datasets/emnabouhajeb/lrs2-v1/mvlrs_v1"   
    TRAIN_SPLIT = "main"          # folder to use for train/val (pretrain is much bigger, optional)
    CKPT_DIR    = "/kaggle/working/checkpoints"
    LOG_DIR     = "/kaggle/working/logs"

    # ---------------- audio ----------------
    SAMPLE_RATE      = 16000
    CHUNK_SECONDS    = 3.0                       # fixed-length audio chunk used per sample
    CHUNK_SAMPLES    = int(SAMPLE_RATE * CHUNK_SECONDS)
    N_FFT            = 512

    # ---------------- video ----------------
    VIDEO_FPS        = 25
    FRAMES_PER_CHUNK = int(VIDEO_FPS * CHUNK_SECONDS)   # 75 frames @ 3s
    LIP_CROP_SIZE    = 96                         # HxW of mouth ROI fed to visual encoder

    # ---------------- audio encoder/decoder (Conv-TasNet style) ----------------
    ENC_KERNEL   = 16
    ENC_STRIDE   = 8
    ENC_CHANNELS = 256                            # N in Conv-TasNet notation

    # ---------------- separator (SepFormer dual-path transformer) ----------------
    SEP_CHUNK_SIZE   = 100      # dual-path local chunk length (in encoder frames)
    SEP_HOP_SIZE     = 50       # 50% overlap
    SEP_D_MODEL      = 256      # kept == ENC_CHANNELS so no extra projection needed
    SEP_N_HEADS      = 4
    SEP_FF_DIM       = 512
    SEP_N_LAYERS     = 4        # 4 dual-path blocks (intra+inter each) -> good accuracy/memory tradeoff
    SEP_DROPOUT      = 0.1

    # ---------------- visual encoder (lightweight AV-HuBERT-style frontend) ----------------
    VIS_EMBED_DIM = 256          # projected to match SEP_D_MODEL for FiLM conditioning

    # ---------------- training ----------------
    BATCH_SIZE        = 4                 # audio+video pairs per step; raise if memory allows
    GRAD_ACCUM_STEPS  = 4                 # effective batch = 16
    NUM_WORKERS       = 2
    EPOCHS            = 10
    LR                = 1.5e-4
    WEIGHT_DECAY      = 1e-5
    WARMUP_STEPS      = 1000
    GRAD_CLIP         = 5.0
    USE_AMP           = True              # mixed precision -> ~2x memory savings
    USE_GRAD_CKPT     = True              # trade compute for memory in separator blocks

    # ---------------- colab/kaggle session budget ----------------
    MAX_TRAIN_SECONDS   = 8.5 * 3600      # stop + checkpoint before the session gets killed
    CKPT_EVERY_STEPS    = 500             # rolling checkpoint (also acts as resume point)
    VAL_EVERY_STEPS     = 500

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED   = 42

    @staticmethod
    def make_dirs():
        os.makedirs(Config.CKPT_DIR, exist_ok=True)
        os.makedirs(Config.LOG_DIR, exist_ok=True)


cfg = Config

Overwriting /kaggle/working/code/config.py


In [7]:
import sys
import config as cfg

# 1. Force the active memory configuration to use the correct path
cfg.Config.DATA_ROOT = "/kaggle/input/datasets/emnabouhajeb/lrs2-v1/mvlrs_v1"

# 2. Wipe the dataset module from cache so it picks up the change
if 'data.lrs2_dataset' in sys.modules:
    del sys.modules['data.lrs2_dataset']

from data.lrs2_dataset import LRS2MixtureDataset

# 3. Run the test
ds = LRS2MixtureDataset(train=True)
print("Success! Dataset size:", len(ds))

Success! Dataset size: 45757


In [8]:
%%writefile /kaggle/working/code/train.py
"""
train.py
Entry point. Run with:  python train.py
Resumes automatically from the latest checkpoint in cfg.CKPT_DIR if present,
so a Colab/Kaggle session that gets killed mid-training can just be re-run.
"""

import os
import time
import glob
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler

from config import cfg
from data.lrs2_dataset import LRS2MixtureDataset, collate_fn
from models.rafravss import RAFRAVSS
from utils.metrics import si_sdr_loss, compute_all_metrics

# Import the evaluation runner from your metrics script
from metrics_eval import run_eval


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_latest_ckpt():
    ckpts = sorted(glob.glob(os.path.join(cfg.CKPT_DIR, "step_*.pt")),
                    key=lambda p: int(p.split("_")[-1].split(".")[0]))
    return ckpts[-1] if ckpts else None


def save_ckpt(model, optimizer, scheduler, scaler, step, epoch, iteration_idx):
    cfg.make_dirs()
    path = os.path.join(cfg.CKPT_DIR, f"step_{step}.pt")
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "step": step,
        "epoch": epoch,
        "iteration_idx": iteration_idx,  # Tracks progress within the current epoch
    }, path)
    # keep only the 3 most recent checkpoints to save disk
    all_ckpts = sorted(glob.glob(os.path.join(cfg.CKPT_DIR, "step_*.pt")),
                        key=lambda p: int(p.split("_")[-1].split(".")[0]))
    for old in all_ckpts[:-3]:
        os.remove(old)
    return path


def lr_lambda(step):
    if step < cfg.WARMUP_STEPS:
        return step / max(1, cfg.WARMUP_STEPS)
    return (cfg.WARMUP_STEPS / max(1, step)) ** 0.5


@torch.no_grad()
def validate(model, val_loader, max_batches=20):
    model.eval()
    all_metrics = {}
    n = 0
    for i, batch in enumerate(val_loader):
        if i >= max_batches:
            break
        mix = batch["mix_audio"].to(cfg.DEVICE)
        target = batch["target_audio"].to(cfg.DEVICE)
        lips = batch["lip_frames"].to(cfg.DEVICE)
        with autocast(enabled=cfg.USE_AMP):
            est = model(mix, lips)
        m = compute_all_metrics(est.float(), target.float(), sr=cfg.SAMPLE_RATE)
        for k, v in m.items():
            all_metrics[k] = all_metrics.get(k, 0.0) + v
        n += 1
    model.train()
    return {k: v / max(1, n) for k, v in all_metrics.items()}


def main():
    set_seed(cfg.SEED)
    cfg.make_dirs()

    train_ds = LRS2MixtureDataset(train=True)
    val_ds = LRS2MixtureDataset(train=False)
    print(f"Train utterances: {len(train_ds)} | Val utterances: {len(val_ds)}")

    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=cfg.NUM_WORKERS, collate_fn=collate_fn,
        pin_memory=True, drop_last=True, persistent_workers=cfg.NUM_WORKERS > 0,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=cfg.NUM_WORKERS, collate_fn=collate_fn,
        pin_memory=True, drop_last=True,
    )

    model = RAFRAVSS().to(cfg.DEVICE)
    print(f"Model params: {model.num_params() / 1e6:.2f}M")

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = GradScaler(enabled=cfg.USE_AMP)

    step = 0
    start_epoch = 0
    resume_iteration = 0
    
    ckpt_path = get_latest_ckpt()
    if ckpt_path is not None:
        print(f"Resuming cleanly from checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=cfg.DEVICE)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        step = ckpt["step"]
        start_epoch = ckpt["epoch"]
        resume_iteration = ckpt.get("iteration_idx", 0)

    train_start_time = time.time()
    model.train()
    stop_training = False

    for epoch in range(start_epoch, cfg.EPOCHS):
        if stop_training:
            break
        optimizer.zero_grad(set_to_none=True)

        for i, batch in enumerate(train_loader):
            # Fast-forward past samples that were already processed in a previous crash
            if resume_iteration > 0 and i < resume_iteration:
                continue

            elapsed = time.time() - train_start_time
            if elapsed > cfg.MAX_TRAIN_SECONDS:
                print(f"Hit MAX_TRAIN_SECONDS budget ({elapsed:.0f}s) -> checkpointing and stopping cleanly.")
                save_ckpt(model, optimizer, scheduler, scaler, step, epoch, i)
                stop_training = True
                break

            mix = batch["mix_audio"].to(cfg.DEVICE, non_blocking=True)
            target = batch["target_audio"].to(cfg.DEVICE, non_blocking=True)
            lips = batch["lip_frames"].to(cfg.DEVICE, non_blocking=True)

            with autocast(enabled=cfg.USE_AMP):
                est = model(mix, lips)
                loss = si_sdr_loss(est.float(), target.float()) / cfg.GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if (i + 1) % cfg.GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1

                if step % 50 == 0:
                    print(f"epoch {epoch} step {step} | loss(-SI-SDR/accum) {loss.item()*cfg.GRAD_ACCUM_STEPS:.3f} "
                            f"| lr {scheduler.get_last_lr()[0]:.2e} | mem {torch.cuda.memory_allocated()/1e9:.2f}GB")

                if step % cfg.CKPT_EVERY_STEPS == 0:
                    save_ckpt(model, optimizer, scheduler, scaler, step, epoch, i)

                if step % cfg.VAL_EVERY_STEPS == 0:
                    metrics = validate(model, val_loader)
                    print(f"[val @ step {step}] " + " ".join(f"{k}={v:.3f}" for k, v in metrics.items()))

        # Reset fast-forward index upon transitioning to a new epoch
        resume_iteration = 0

    print("Training finished (or paused for session limit). Final checkpoint saved in", cfg.CKPT_DIR)

    # ---------------- AUTOMATIC EVALUATION BLOCK ----------------
    print("\n==================================================")
    print("[Auto-Eval] Training loop terminated. Launching complete metric validation...")
    print("==================================================")
    
    try:
        eval_ds = LRS2MixtureDataset(train=False)
        eval_loader = DataLoader(
            eval_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
            num_workers=cfg.NUM_WORKERS, collate_fn=collate_fn, drop_last=False
        )
        
        model.eval()
        summary, csv_path, summary_path, audio_dir, n_saved = run_eval(
            model=model,
            loader=eval_loader,
            device=cfg.DEVICE,
            max_samples=500,
            save_audio_n=10,
            out_dir="/kaggle/working/eval_results"
        )
        
        print("\n===== Evaluation summary =====")
        for k, v in summary.items():
            print(f"{k:10s} : {v['mean']:.3f} +/- {v['std']:.3f}")
        print(f"\nPer-sample metrics saved to : {csv_path}")
        print(f"Summary data saved to       : {summary_path}")
        print(f"Audio wav triplets saved to : {audio_dir}")
        
    except Exception as e:
        print(f"\n[Auto-Eval Warning] Evaluation script encountered an error: {e}")


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/code/train.py


In [9]:
!python train.py

Train utterances: 45757 | Val utterances: 2408
Model params: 5.21M
/kaggle/working/code/train.py:110: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.USE_AMP)
/kaggle/working/code/train.py:153: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.USE_AMP):
/kaggle/working/code/train.py:164: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
epoch 0 step 50 | loss(-SI-SDR/accum) 4.286 | lr 7.50e-06 | mem 0.09GB
epoch 0 step 100 